## 📦 Kütüphaneleri İçe Aktar

In [1]:
import numpy as np
import pandas as pd
import json
import pickle
import warnings

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, classification_report
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# ML Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 📂 Ön İşlenmiş Veriyi Yükle

In [2]:
# Kaggle input directory (from added notebook 3)
# NOT: Notebook 3'ü "Add Data" ile eklediyseniz path bu şekildedir
data_dir = '/kaggle/input/3-feature-engineering-notebook/'

# Load data
X_train = np.load(data_dir + 'X_train_fe.npy')
X_test = np.load(data_dir + 'X_test_fe.npy')
y_train = np.load(data_dir + 'y_train.npy')
y_test = np.load(data_dir + 'y_test.npy')

# Load feature names
with open(data_dir + 'feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]

print("✅ Data loaded successfully!")
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nNumber of features: {len(feature_names)}")

✅ Data loaded successfully!

X_train shape: (8000, 19)
X_test shape: (2000, 19)
y_train shape: (8000,)
y_test shape: (2000,)

Number of features: 19


## 🎯 Hiperparametre Gridlerini Tanımla

In [3]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    """
    Evaluate model and return metrics
    """
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Probabilities
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_test_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    results = {
        'train': {
            'accuracy': accuracy_score(y_train, y_train_pred),
            'precision': precision_score(y_train, y_train_pred),
            'recall': recall_score(y_train, y_train_pred),
            'f1': f1_score(y_train, y_train_pred),
            'roc_auc': roc_auc_score(y_train, y_train_proba)
        },
        'test': {
            'accuracy': accuracy_score(y_test, y_test_pred),
            'precision': precision_score(y_test, y_test_pred),
            'recall': recall_score(y_test, y_test_pred),
            'f1': f1_score(y_test, y_test_pred),
            'roc_auc': roc_auc_score(y_test, y_test_proba)
        }
    }
    
    return results

print("Helper function defined!")

Helper function defined!


## 🤖 Model 1: XGBoost ile RandomizedSearchCV

In [4]:
print("Training XGBoost with RandomizedSearchCV...\n")

# Define hyperparameter space
xgb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

# Base model
xgb_base = XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)

# RandomizedSearchCV
xgb_random = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_grid,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Fit
xgb_random.fit(X_train, y_train)

# Best model
xgb_best = xgb_random.best_estimator_

print(f"\n✅ XGBoost training completed!")
print(f"Best CV ROC-AUC: {xgb_random.best_score_:.4f}")
print(f"Best parameters: {xgb_random.best_params_}")

# Evaluate
xgb_results = evaluate_model(xgb_best, X_train, X_test, y_train, y_test)
print(f"\nTest ROC-AUC: {xgb_results['test']['roc_auc']:.4f}")
print(f"Test F1-Score: {xgb_results['test']['f1']:.4f}")

Training XGBoost with RandomizedSearchCV...

Fitting 5 folds for each of 20 candidates, totalling 100 fits

✅ XGBoost training completed!
Best CV ROC-AUC: 0.8654
Best parameters: {'subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.6}

Test ROC-AUC: 0.8670
Test F1-Score: 0.5831


## 🤖 Model 2: LightGBM ile RandomizedSearchCV

In [5]:
print("Training LightGBM with RandomizedSearchCV...\n")

# Define hyperparameter space
lgbm_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 9, -1],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [31, 50, 70, 100],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_samples': [10, 20, 30]
}

# Base model
lgbm_base = LGBMClassifier(random_state=42, verbose=-1)

# RandomizedSearchCV
lgbm_random = RandomizedSearchCV(
    estimator=lgbm_base,
    param_distributions=lgbm_param_grid,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Fit
lgbm_random.fit(X_train, y_train)

# Best model
lgbm_best = lgbm_random.best_estimator_

print(f"\n✅ LightGBM training completed!")
print(f"Best CV ROC-AUC: {lgbm_random.best_score_:.4f}")
print(f"Best parameters: {lgbm_random.best_params_}")

# Evaluate
lgbm_results = evaluate_model(lgbm_best, X_train, X_test, y_train, y_test)
print(f"\nTest ROC-AUC: {lgbm_results['test']['roc_auc']:.4f}")
print(f"Test F1-Score: {lgbm_results['test']['f1']:.4f}")

Training LightGBM with RandomizedSearchCV...

Fitting 5 folds for each of 20 candidates, totalling 100 fits

✅ LightGBM training completed!
Best CV ROC-AUC: 0.8644
Best parameters: {'subsample': 0.8, 'num_leaves': 100, 'n_estimators': 100, 'min_child_samples': 30, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

Test ROC-AUC: 0.8717
Test F1-Score: 0.5902


## 🤖 Model 3: CatBoost ile RandomizedSearchCV

In [6]:
print("Training CatBoost with RandomizedSearchCV...\n")

# Define hyperparameter space
catboost_param_grid = {
    'iterations': [100, 200, 300, 500],
    'depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7],
    'border_count': [32, 64, 128],
    'bagging_temperature': [0, 0.5, 1]
}

# Base model
catboost_base = CatBoostClassifier(random_state=42, verbose=0)

# RandomizedSearchCV
catboost_random = RandomizedSearchCV(
    estimator=catboost_base,
    param_distributions=catboost_param_grid,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Fit
catboost_random.fit(X_train, y_train)

# Best model
catboost_best = catboost_random.best_estimator_

print(f"\n✅ CatBoost training completed!")
print(f"Best CV ROC-AUC: {catboost_random.best_score_:.4f}")
print(f"Best parameters: {catboost_random.best_params_}")

# Evaluate
catboost_results = evaluate_model(catboost_best, X_train, X_test, y_train, y_test)
print(f"\nTest ROC-AUC: {catboost_results['test']['roc_auc']:.4f}")
print(f"Test F1-Score: {catboost_results['test']['f1']:.4f}")

Training CatBoost with RandomizedSearchCV...

Fitting 5 folds for each of 20 candidates, totalling 100 fits

✅ CatBoost training completed!
Best CV ROC-AUC: 0.8684
Best parameters: {'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 100, 'depth': 5, 'border_count': 64, 'bagging_temperature': 0}

Test ROC-AUC: 0.8741
Test F1-Score: 0.6062


## 📊 Model Performanslarını Karşılaştır

In [7]:
# Compile all results
all_results = {
    'XGBoost': xgb_results,
    'LightGBM': lgbm_results,
    'CatBoost': catboost_results
}

# Create comparison dataframe
comparison_data = []
for model_name, results in all_results.items():
    comparison_data.append({
        'Model': model_name,
        'Train ROC-AUC': results['train']['roc_auc'],
        'Test ROC-AUC': results['test']['roc_auc'],
        'Train F1': results['train']['f1'],
        'Test F1': results['test']['f1'],
        'Test Accuracy': results['test']['accuracy'],
        'Test Precision': results['test']['precision'],
        'Test Recall': results['test']['recall']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Test ROC-AUC', ascending=False)

print("\n" + "="*80)
print("📊 MODEL COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)


📊 MODEL COMPARISON
   Model  Train ROC-AUC  Test ROC-AUC  Train F1  Test F1  Test Accuracy  Test Precision  Test Recall
CatBoost       0.882767      0.874127  0.598081 0.606154          0.872        0.810700     0.484029
LightGBM       0.885760      0.871686  0.602992 0.590214          0.866        0.781377     0.474201
 XGBoost       0.905188      0.866958  0.620233 0.583072          0.867        0.805195     0.457002


## 🏆 En İyi Modeli Seç

In [8]:
# Find best model based on Test ROC-AUC
best_model_name = comparison_df.iloc[0]['Model']
best_test_roc_auc = comparison_df.iloc[0]['Test ROC-AUC']

# Get best model object
model_mapping = {
    'XGBoost': (xgb_best, xgb_random.best_params_),
    'LightGBM': (lgbm_best, lgbm_random.best_params_),
    'CatBoost': (catboost_best, catboost_random.best_params_)
}

best_model, best_params = model_mapping[best_model_name]

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"Test ROC-AUC: {best_test_roc_auc:.4f}")
print(f"\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"  {param}: {value}")


🏆 BEST MODEL: CatBoost
Test ROC-AUC: 0.8741

Best Hyperparameters:
  learning_rate: 0.1
  l2_leaf_reg: 5
  iterations: 100
  depth: 5
  border_count: 64
  bagging_temperature: 0


## 💾 En İyi Modeli Kaydet

In [9]:
# Output directory
output_dir = '/kaggle/working/'

# Save best model
with open(output_dir + 'best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save all results
with open(output_dir + 'results.json', 'w') as f:
    json.dump(all_results, f, indent=4)

# Save best parameters
with open(output_dir + 'best_params.json', 'w') as f:
    json.dump({
        'best_model': best_model_name,
        'best_params': best_params,
        'test_roc_auc': float(best_test_roc_auc)
    }, f, indent=4)

print("✅ All files saved successfully!")
print(f"\nSaved files:")
print(f"  - best_model.pkl ({best_model_name})")
print(f"  - results.json (all model results)")
print(f"  - best_params.json (best hyperparameters)")

✅ All files saved successfully!

Saved files:
  - best_model.pkl (CatBoost)
  - results.json (all model results)
  - best_params.json (best hyperparameters)


## 📝 Model Optimizasyonu Özeti

### Eğitilen Modeller:

#### 1. XGBoost
- **Arama Tipi:** RandomizedSearchCV (20 iterasyon, 5-fold CV)
- **En İyi Parametreler:** ['subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.6]
- **Test ROC-AUC:** [0.8670]

#### 2. LightGBM
- **Arama Tipi:** RandomizedSearchCV (20 iterasyon, 5-fold CV)
- **En İyi Parametreler:** ['subsample': 0.8, 'num_leaves': 100, 'n_estimators': 100, 'min_child_samples': 30, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.8]
- **Test ROC-AUC:** [0.8717]

#### 3. CatBoost
- **Arama Tipi:** RandomizedSearchCV (20 iterasyon, 5-fold CV)
- **En İyi Parametreler:** ['learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 100, 'depth': 5, 'border_count': 64, 'bagging_temperature': 0]
- **Test ROC-AUC:** [0.8741]

### En İyi Model:
**[CatBoost]** - ROC-AUC: [0.8741]

### Kaydedilen Dosyalar:
✅ `best_model.pkl` - En iyi model
✅ `model_results.json` - Tüm model sonuçları

### Sonraki Adımlar:
1. **Model Değerlendirme:** SHAP değerleri, confusion matrix, ROC curves
2. **Pipeline Oluşturma:** Tam ön işleme + model pipeline'ı
3. **Deployment:** Streamlit uygulaması için pipeline'ı paketleme